In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.inference import VariableElimination

data = pd.DataFrame({
    'age': pd.cut(df['age'], [0, 45, 55, 100], labels=[0, 1, 2]).astype(int),
    'sex': df['sex'],
    'cp': df['cp'], 
    'bp': pd.cut(df['trestbps'], [0, 120, 140, 300], labels=[0, 1, 2]).astype(int),
    'chol': pd.cut(df['chol'], [0, 200, 240, 700], labels=[0, 1, 2]).astype(int),
    'thalach': pd.cut(df['thalach'], [0, 120, 150, 250], labels=[0, 1, 2]).astype(int),
    'exang': df['exang'],
    'oldpeak': pd.cut(df['oldpeak'], [-1, 0.5, 2, 10], labels=[0, 1, 2]).astype(int),
    'ca': df['ca'], # major vessels 0-4
    'thal': df['thal'], # thalassemia 0-3
    'disease': df['disease'],
})
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['disease'])
edges = [
    ('age', 'bp'), ('age', 'chol'), ('age', 'disease'), ('sex', 'disease'),
    ('bp', 'disease'), ('chol', 'disease'),
    ('disease', 'cp'), ('disease', 'exang'), ('disease', 'thalach'),
    ('disease', 'oldpeak'), ('disease', 'ca'), ('disease', 'thal')
]
model = DiscreteBayesianNetwork(edges)
model.fit(train, estimator=DiscreteBayesianEstimator())
print("Model valid:", model.check_model())
print("\nLearned CPT of chest pain (cp) given disease:\n")
print(model.get_cpds('cp'))

infer = VariableElimination(model)
print("\nDiagnosis queries")
q1 = infer.query(['disease'], evidence={'age': 2, 'sex': 1, 'cp': 0, 'exang': 1})
print("P(disease | old, male, asymptomatic chest pain, exercise angina):\n", q1)
q2 = infer.query(['disease'], evidence={'age': 0, 'sex': 0, 'cp': 2, 'exang': 0, 'thalach': 2})
print("P(disease | young, female, non-anginal pain, no angina, high max HR):\n", q2)
evidence_cols = ['age', 'sex', 'bp', 'chol', 'cp', 'exang', 'thalach', 'oldpeak', 'ca', 'thal']
pred = []
for _, row in test.iterrows():
    p = infer.query(['disease'], evidence=row[evidence_cols].to_dict(), show_progress=False)
    pred.append(int(p.values[1] > 0.5))
acc = (pd.Series(pred, index=test.index) == test['disease']).mean()
print(f"\nTest accuracy: {acc:.3f} ({len(test)} patients)")
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(test['disease'], pred))
print(classification_report(test['disease'], pred, target_names=['No disease', 'Disease']))

Model valid: True

Learned CPT of chest pain (cp) given disease:

+---------+---------------------+----------------------+
| disease | disease(0)          | disease(1)           |
+---------+---------------------+----------------------+
| cp(0)   | 0.2425650557620818  | 0.7522222222222222   |
+---------+---------------------+----------------------+
| cp(1)   | 0.2648698884758364  | 0.058888888888888886 |
+---------+---------------------+----------------------+
| cp(2)   | 0.3912639405204461  | 0.13                 |
+---------+---------------------+----------------------+
| cp(3)   | 0.10130111524163568 | 0.058888888888888886 |
+---------+---------------------+----------------------+

Diagnosis queries
P(disease | old, male, asymptomatic chest pain, exercise angina):
 +------------+----------------+
| disease    |   phi(disease) |
+============+================+
| disease(0) |         0.0356 |
+------------+----------------+
| disease(1) |         0.9644 |
+------------+---------------